# CoNSeP 数据集处理
本 notebook 将 consep_raw 中的实例分割标注转换为 Faster R-CNN 可用的 CSV 检测标注。
处理逻辑与 MCSpatNet 的 CoNSeP 三类映射保持一致：1 = inflammatory，2 = epithelial（3+4），3 = stromal / spindle（1+5+6+7）。
不做 patch 裁剪，直接以原始整张图像为单位导出 images、annotations/boxes.csv 和 metadata/classes.csv。
Train 目录会按照 data_splits/consep/train_split.txt 与 val_split.txt 划分为 train / val，Test 目录直接作为 test。

In [1]:
from __future__ import annotations

import csv
import shutil
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from tqdm.auto import tqdm

SOURCE_ROOT = Path("consep_raw")
OUTPUT_ROOT = Path("data") / "CoNSeP"
TRAIN_SPLIT_PATH = Path("..") / "data_splits" / "consep" / "train_split.txt"
VAL_SPLIT_PATH = Path("..") / "data_splits" / "consep" / "val_split.txt"
OVERWRITE_OUTPUT = False

CLASS_GROUP_MAPPING_DICT = {1: [2], 2: [3, 4], 3: [1, 5, 6, 7]}
CLASS_NAMES = {
    1: "inflammatory",
    2: "epithelial",
    3: "stromal",
}

LABEL_TO_GROUP = {
    source_label: group_label
    for group_label, source_labels in CLASS_GROUP_MAPPING_DICT.items()
    for source_label in source_labels
}

e:\WYH\CV\MCSpatNet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_split_stems(split_path: Path) -> set[str]:
    stems: set[str] = set()
    with split_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            name = line.strip()
            if not name:
                continue
            stems.add(Path(name).stem)
    return stems


def ensure_output_root(output_root: Path, overwrite: bool) -> None:
    if output_root.exists() and overwrite:
        shutil.rmtree(output_root)

    (output_root / "images" / "train").mkdir(parents=True, exist_ok=True)
    (output_root / "images" / "val").mkdir(parents=True, exist_ok=True)
    (output_root / "images" / "test").mkdir(parents=True, exist_ok=True)
    (output_root / "annotations").mkdir(parents=True, exist_ok=True)
    (output_root / "metadata").mkdir(parents=True, exist_ok=True)


def resolve_train_split(sample_stem: str, train_stems: set[str], val_stems: set[str]) -> str:
    if sample_stem in train_stems:
        return "train"
    if sample_stem in val_stems:
        return "val"
    raise ValueError(f"{sample_stem} 不在 train/val split 文件中")


def extract_boxes_from_mat(mat_path: Path) -> list[dict[str, int | str]]:
    mat = loadmat(mat_path)
    inst_map = mat["inst_map"]
    inst_type = np.asarray(mat["inst_type"]).reshape(-1)

    boxes: list[dict[str, int | str]] = []
    instance_ids = np.unique(inst_map)
    instance_ids = instance_ids[instance_ids > 0]

    for instance_id in instance_ids:
        instance_index = int(instance_id) - 1
        if instance_index < 0 or instance_index >= len(inst_type):
            continue

        raw_label = int(inst_type[instance_index])
        group_label = LABEL_TO_GROUP.get(raw_label)
        if group_label is None:
            continue

        ys, xs = np.where(inst_map == instance_id)
        if ys.size == 0 or xs.size == 0:
            continue

        xmin = int(xs.min())
        ymin = int(ys.min())
        xmax = int(xs.max()) + 1
        ymax = int(ys.max()) + 1

        if xmax <= xmin or ymax <= ymin:
            continue

        boxes.append(
            {
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "label": CLASS_NAMES[group_label],
                "label_id": group_label,
            }
        )

    return boxes

In [3]:
def export_split(
    image_dir: Path,
    label_dir: Path,
    split_name: str,
    output_root: Path,
    rows: list[dict[str, int | str]],
    known_stems: set[str] | None = None,
 ) -> dict[str, int]:
    image_paths = sorted(image_dir.glob("*.png"))
    image_paths += sorted(image_dir.glob("*.jpg"))
    image_paths += sorted(image_dir.glob("*.jpeg"))
    image_paths += sorted(image_dir.glob("*.tif"))
    image_paths += sorted(image_dir.glob("*.tiff"))

    image_count = 0
    box_count = 0

    for image_path in tqdm(image_paths, desc=f"导出 {split_name}", unit="image"):
        sample_stem = image_path.stem
        if known_stems is not None and sample_stem not in known_stems:
            continue

        mat_path = label_dir / f"{sample_stem}.mat"
        if not mat_path.exists():
            raise FileNotFoundError(f"找不到与图像对应的标注文件: {mat_path}")

        destination = output_root / "images" / split_name / image_path.name
        if destination.resolve() != image_path.resolve():
            shutil.copy2(image_path, destination)

        relative_image_path = Path("images") / split_name / image_path.name
        boxes = extract_boxes_from_mat(mat_path)
        image_count += 1

        if not boxes:
            rows.append(
                {
                    "split": split_name,
                    "image_path": relative_image_path.as_posix(),
                    "xmin": "",
                    "ymin": "",
                    "xmax": "",
                    "ymax": "",
                    "label": "background",
                    "label_id": 0,
                    "is_negative": 1,
                }
            )
            continue

        for box in boxes:
            rows.append(
                {
                    "split": split_name,
                    "image_path": relative_image_path.as_posix(),
                    "xmin": box["xmin"],
                    "ymin": box["ymin"],
                    "xmax": box["xmax"],
                    "ymax": box["ymax"],
                    "label": box["label"],
                    "label_id": box["label_id"],
                    "is_negative": 0,
                }
            )
            box_count += 1

    return {"images": image_count, "boxes": box_count}

In [4]:
def prepare_consep_detection_dataset(
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    train_split_path: Path = TRAIN_SPLIT_PATH,
    val_split_path: Path = VAL_SPLIT_PATH,
    overwrite: bool = OVERWRITE_OUTPUT,
 ) -> None:
    source_root = Path(source_root)
    output_root = Path(output_root)
    train_split_path = Path(train_split_path)
    val_split_path = Path(val_split_path)

    ensure_output_root(output_root, overwrite)

    train_stems = read_split_stems(train_split_path)
    val_stems = read_split_stems(val_split_path)
    overlap = train_stems & val_stems
    if overlap:
        raise ValueError(f"train/val split 存在重复样本: {sorted(overlap)[:5]}")

    rows: list[dict[str, int | str]] = []

    train_dir = source_root / "Train"
    test_dir = source_root / "Test"

    train_images_dir = train_dir / "Images"
    train_labels_dir = train_dir / "Labels"
    test_images_dir = test_dir / "Images"
    test_labels_dir = test_dir / "Labels"

    summary_train = export_split(
        train_images_dir,
        train_labels_dir,
        "train",
        output_root,
        rows,
        known_stems=train_stems,
    )
    summary_val = export_split(
        train_images_dir,
        train_labels_dir,
        "val",
        output_root,
        rows,
        known_stems=val_stems,
    )
    summary_test = export_split(
        test_images_dir,
        test_labels_dir,
        "test",
        output_root,
        rows,
    )

    fieldnames = [
        "split",
        "image_path",
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "label",
        "label_id",
        "is_negative",
    ]

    with (output_root / "annotations" / "boxes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    with (output_root / "metadata" / "classes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["label_id", "label"])
        writer.writeheader()
        writer.writerow({"label_id": 0, "label": "background"})
        for label_id, label_name in CLASS_NAMES.items():
            writer.writerow({"label_id": label_id, "label": label_name})

    print(f"输出目录: {output_root.resolve()}")
    print(
        "split 统计:",
        {
            "train": summary_train,
            "val": summary_val,
            "test": summary_test,
        },
    )
    print(f"总标注框数: {sum(int(row['is_negative']) == 0 for row in rows)}")

In [5]:
# 按需修改上面的参数后，再运行本单元。

prepare_consep_detection_dataset(
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    train_split_path=TRAIN_SPLIT_PATH,
    val_split_path=VAL_SPLIT_PATH,
    overwrite=OVERWRITE_OUTPUT,
)

导出 test: 100%|██████████| 14/14 [00:25<00:00,  1.83s/image]

输出目录: E:\WYH\CV\MCSpatNet\contrast\data\CoNSeP
split 统计: {'train': {'images': 22, 'boxes': 13040}, 'val': {'images': 5, 'boxes': 2515}, 'test': {'images': 14, 'boxes': 8777}}
总标注框数: 24332
